# 03 — Filtering Spasial, Konvolusi, dan Karakteristik Sistem Pencitraan
**Pertanyaan utama:** ketika filter membuat citra lebih halus atau lebih tajam, informasi apa yang dipertahankan, informasi apa yang hilang, dan apakah hasil pengukuran menjadi lebih dapat dipercaya?

Notebook ini mengembangkan hubungan antara operasi spasial dan model fisika pembentukan citra. Kita akan menurunkan konvolusi, memeriksa respons sistem linear, menghubungkan *point spread function* (PSF) dengan *optical transfer function* (OTF), memodelkan noise, serta menguji kapan filter memperbaiki tugas pengukuran—dan kapan justru menyebabkan bias.

**Prasyarat:** Notebook 01 (representasi, sensor, sampling, kuantisasi) dan 02 (histogram, kontras, normalisasi); dasar aljabar linear, probabilitas, dan NumPy. **Alokasi:** 3–4 sesi kuliah/praktikum ditambah studi kasus mandiri.

> **Provenance data:** `camera` berasal dari dataset contoh `scikit-image`. Model sensor, PSF, variasi spesimen, cacat permukaan, dan noise pada studi kasus merupakan **simulasi sintetis**, bukan bukti performa perangkat optik nyata. Semua proses dapat dijalankan lokal dan tidak memerlukan jaringan atau GPU.

## Peta kompetensi dan pertanyaan pemandu

Setelah materi ini, pembaca diharapkan mampu:

1. Membedakan peningkatan kontras global dengan operasi yang memanfaatkan hubungan spasial.
2. Menurunkan konvolusi dua dimensi, membedakannya dari korelasi, serta menjelaskan *kernel*, normalisasi, dan perlakuan batas.
3. Menguji linearitas dan invarian pergeseran, termasuk ketika asumsi tersebut gagal di dekat batas citra.
4. Menurunkan sifat Gaussian, *separability*, dan dampaknya terhadap PSF serta MTF.
5. Memilih filter menurut model noise dan sifat objek, bukan berdasarkan tampilan semata.
6. Menurunkan varians noise setelah filtering dan mengenali korelasi antarpiksel keluarannya.
7. Menguji efek penghalusan dan *sharpening* terhadap tepi, detail kecil, dan ketidakpastian pengukuran.
8. Membandingkan pendekatan filtering memakai protokol validasi terpisah dari data uji dan metrik yang sesuai.

**Pertanyaan pemandu:** Mengapa citra dengan PSNR lebih tinggi dapat menghasilkan pengukuran diameter yang lebih buruk? Mengapa rata-rata banyak piksel yang sudah difilter tidak otomatis memiliki ketidakpastian sekecil yang dihitung dengan asumsi piksel independen?

## 1. Persiapan: kontrak numerik, satuan, dan visualisasi

Citra contoh akan dinormalisasi pada rentang nilai kode `[0,1]`. **Jangan** langsung mengartikan angka ini sebagai radiansi atau reflektansi fisis. Pada *phantom* sintetis, nilai intensitas dibangun sebagai besaran proporsional terhadap sinyal terukur, dengan asumsi sensor linear sebelum *clipping*.

Gunakan koordinat array $I[m,n]$ dengan $m$ sebagai indeks baris dan $n$ indeks kolom. Frekuensi spasial dilaporkan dalam **siklus/piksel**; dalam eksperimen geometri, skala hipotetis dinyatakan dalam mm/piksel. Bandingkan citra dengan `vmin/vmax` tetap agar *autoscaling* tidak menyesatkan.

In [ ]:
import platform, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, signal
from skimage import data, filters, restoration, metrics, measure
from skimage.util import img_as_float
from IPython.display import display

SEED = 202603
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':110, 'axes.grid':False, 'font.size':10})

def lihat(ax, image, title, vmin=0, vmax=1):
    ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')

def skor_psnr_ssim(ref, pred):
    return (metrics.peak_signal_noise_ratio(ref,pred,data_range=1.0),
            metrics.structural_similarity(ref,pred,data_range=1.0))

camera = img_as_float(data.camera())[96:352,96:352]
print('Python:',platform.python_version(), '| NumPy:',np.__version__)
print('Citra:',camera.shape,'| rentang:',np.round([camera.min(),camera.max()],3))
fig, ax=plt.subplots(1,2,figsize=(9,3.4))
lihat(ax[0],camera,'Citra referensi (kode dinormalisasi)')
ax[1].plot(camera[125,:],lw=1);ax[1].set(xlabel='Kolom (piksel)',ylabel='Nilai kode [0,1]',title='Profil intensitas baris ke-125');ax[1].grid(alpha=.2)
plt.tight_layout();plt.show()

## 2. Mengapa filtering tidak sama dengan transformasi histogram?

Histogram dari Notebook 02 tidak menyimpan susunan piksel; dua citra hasil permutasi posisi dapat mempunyai histogram identik. Filtering spasial menggunakan **kedekatan dan susunan** piksel. Dengan demikian, respons terhadap tepi, tekstur, dan pola periodik sangat bergantung pada geometri citra—bahkan jika histogram globalnya tidak berubah.

Model awal pengukuran yang lebih realistis daripada $g=f+\eta$ adalah

$$
g[m,n]=(h*f)[m,n]+\eta[m,n],
$$

di mana $f$ menyatakan distribusi intensitas laten pada grid terpilih, $h$ adalah PSF efektif (optik, integrasi piksel, atau gerak dalam aproksimasi konvolusional), dan $\eta$ noise. Model ini mengasumsikan sistem lokal linear dan *shift-invariant*, serta pemrosesan dalam domain sinyal linear. Jika noise dominan berupa shot noise, varians kondisionalnya bergantung pada ekspektasi sinyal; bentuk aditif Gaussian dengan varians konstan hanya pendekatan tertentu.

Filtering pascaakuisisi $u=k*g$ menghasilkan respons efektif $(k*h)*f+k*\eta$, bukan sekadar "menghilangkan noise". Kernel $k$ memodifikasi sinyal **dan** noise sekaligus.

In [ ]:
# Histogram sama, struktur dan keputusan spasial berbeda
xx=np.zeros((128,128),float)
xx[34:95,43:88]=.78;xx[56:72,52:76]=.23
perm=rng.permutation(xx.ravel()).reshape(xx.shape)
ker=np.ones((7,7))/49
sm_x=ndimage.convolve(xx,ker,mode='reflect')
sm_p=ndimage.convolve(perm,ker,mode='reflect')
fig,axs=plt.subplots(2,3,figsize=(11,6))
for a,z,t in zip(axs[0],[xx,perm,np.abs(sm_x-sm_p)],['Objek tersusun','Permutasi piksel','|Hasil filtering berbeda|']):lihat(a,z,t)
axs[1,0].hist(xx.ravel(),bins=30,alpha=.65,label='asli');axs[1,0].hist(perm.ravel(),bins=30,histtype='step',lw=2,label='permutasi');axs[1,0].legend();axs[1,0].set(title='Histogram sama',xlabel='Intensitas')
axs[1,1].plot(sm_x[64],label='asli');axs[1,1].plot(sm_p[64],label='permutasi');axs[1,1].legend();axs[1,1].set(xlabel='Kolom',ylabel='Intensitas',title='Profil hasil filtering')
axs[1,2].axis('off');axs[1,2].text(.03,.88,'Histogram tidak memuat\ninformasi ketetanggaan.\n\nFiltering bekerja pada\nstruktur spasial.',va='top',fontsize=12)
plt.tight_layout();plt.show()
print('Selisih histogram (maksimum):',np.max(np.abs(np.histogram(xx,bins=30,range=(0,1))[0]-np.histogram(perm,bins=30,range=(0,1))[0])))

## 3. Sistem linear, invarian pergeseran, dan respons impuls

Untuk operator $\mathcal H$, **linearitas** mensyaratkan

$$
\mathcal H(af_1+bf_2)=a\mathcal H(f_1)+b\mathcal H(f_2).
$$

**Invarian pergeseran** berarti menggeser input akan menggeser output dengan besaran sama, setidaknya pada domain yang konsisten. Bila kedua sifat dipenuhi, respons terhadap impuls diskret $\delta[m,n]$ menentukan respons terhadap setiap sinyal:

$$
f[m,n]=\sum_{i,j}f[i,j]\,\delta[m-i,n-j]
\quad\Longrightarrow\quad
\mathcal H\{f\}[m,n]=\sum_{i,j}f[i,j]\,h[m-i,n-j].
$$

Di sini $h=\mathcal H\{\delta\}$ adalah respons impuls atau PSF. **Batas gambar** dapat merusak invarian pergeseran efektif: pilihan padding menciptakan asumsi sinyal di luar bidang pandang, bukan fakta pengukuran. Median dan bilateral filtering tidak linear karena bobot/operatornya bergantung pada data.

In [ ]:
# Uji linearitas pada filter linear dan ketidaklinearannya pada median
z1=rng.normal(size=(64,64));z2=rng.normal(size=(64,64)); a,b=.4,-1.3
K=np.array([[0,1,0],[2,3,1],[0,-2,0]],dtype=float)/5
L=lambda z: ndimage.convolve(z,K,mode='wrap')
N=lambda z: ndimage.median_filter(z,size=3,mode='wrap')
err_linear=np.max(np.abs(L(a*z1+b*z2)-(a*L(z1)+b*L(z2))))
err_median=np.max(np.abs(N(a*z1+b*z2)-(a*N(z1)+b*N(z2))))
print('Error linearitas (konvolusi):',f'{err_linear:.2e}')
print('Error linearitas (median):',f'{err_median:.3f}')
# Invarian pergeseran pada domain periodik (mode wrap)
x=np.zeros((64,64));x[25,30]=1
s=(5,-7)
err_shift=np.max(np.abs(L(np.roll(x,s,axis=(0,1)))-np.roll(L(x),s,axis=(0,1))))
print('Error invarian pergeseran (wrap):',f'{err_shift:.2e}')

## 4. Konvolusi 2D, korelasi, dan orientasi kernel

Dengan kernel berukuran ganjil dan berpusat di nol, konvolusi diskret dapat dinyatakan

$$
(k*f)[m,n]=\sum_{i=-r}^{r}\sum_{j=-s}^{s}k[i,j]f[m-i,n-j].
$$

Sementara itu, korelasi silang untuk kernel real adalah

$$
(k\star f)[m,n]=\sum_{i=-r}^{r}\sum_{j=-s}^{s}k[i,j]f[m+i,n+j].
$$

Keduanya sama untuk kernel simetris $k[i,j]=k[-i,-j]$, tetapi dapat berbeda untuk kernel gradien yang asimetris. Dalam machine learning, operasi yang disebut lapisan "convolution" pada banyak pustaka sebenarnya memakai korelasi silang saat *forward pass*.

Untuk filter perataan, $\sum_{i,j}k[i,j]=1$ mempertahankan masukan konstan di interior atau dengan ekstensi batas yang konsisten. Kernel turunan biasanya menjumlah ke nol karena respons terhadap citra konstan harus nol. **Penjumlahan bobot satu tidak menjamin filter mempertahankan tepi, tekstur, ataupun nilai maksimum lokal.**

In [ ]:
# Konvolusi eksplisit untuk memahami pembalikan kernel
f=np.arange(1,26,dtype=float).reshape(5,5)
k=np.array([[0,1,2],[0,0,0],[-1,-2,-3]],float)
def convolution_naif(x,ker):
    py,px=np.array(ker.shape)//2
    X=np.pad(x,((py,py),(px,px)),mode='constant')
    Y=np.zeros_like(x,dtype=float)
    for m in range(x.shape[0]):
        for n in range(x.shape[1]):
            Y[m,n]=np.sum(X[m:m+ker.shape[0],n:n+ker.shape[1]]*ker[::-1,::-1])
    return Y
naive=convolution_naif(f,k)
conv=signal.convolve2d(f,k,mode='same',boundary='fill')
corr=signal.correlate2d(f,k,mode='same',boundary='fill')
print('Kernel asimetris:\n',k)
print('Max |manual - scipy convolve2d|:',np.max(np.abs(naive-conv)))
print('Nilai pusat: konvolusi=',conv[2,2],' korelasi=',corr[2,2])
assert np.allclose(naive,conv)

## 5. Boundary conditions: tepi citra bukan ruang kosong yang diketahui

Operator berbasis jendela membutuhkan sampel di luar citra. Empat kondisi lazim: `constant` (sering nol), `reflect` (pantulan), `nearest` (replikasi piksel tepi), dan `wrap` (periodik). Masing-masing menyatakan **hipotesis ekstensi** yang berbeda.

Pada citra pengukuran, padding nol dapat menghasilkan penurunan intensitas buatan di batas; `wrap` dapat menciptakan interaksi semu antara sisi kiri dan kanan. `reflect` sering praktis untuk mengurangi artefak, namun tetap bukan kondisi batas fisik universal. Ketika hasil dekat tepi penting, hitung *valid region* atau uji sensitivitas terhadap pilihan kondisi batas.

In [ ]:
ims=np.ones((70,90),float)*.75
ims[18:60,12:55]=.3
names=['constant','reflect','nearest','wrap']
fig,ax=plt.subplots(1,5,figsize=(13,2.7))
lihat(ax[0],ims,'Masukan')
for i,mode in enumerate(names,1):
    z=ndimage.gaussian_filter(ims,4,mode=mode)
    lihat(ax[i],z,mode)
    print(f'{mode:<9} nilai sudut: {z[0,0]:.4f}')
plt.tight_layout();plt.show()

## 6. Gaussian filtering: dari difusi ke kernel yang terpisah

PSF Gaussian ideal kontinu dua dimensi berbentuk

$$
k_\sigma(x,y)=\frac{1}{2\pi\sigma^2}\exp\!\left[-\frac{x^2+y^2}{2\sigma^2}\right],\qquad \sigma>0.
$$

Kernel ini dapat dipisahkan: $k_\sigma(x,y)=g_\sigma(x)g_\sigma(y)$, dengan $g_\sigma(t)=\exp[-t^2/(2\sigma^2)]/(\sqrt{2\pi}\sigma)$. Jadi filtering 2D dapat dihitung sebagai dua operasi 1D. Untuk citra berukuran $MN$ dan kernel sisi $K$, operasi langsung memerlukan sekitar $O(MNK^2)$ perkalian, sementara pemisahan memberi sekitar $O(MNK)$—bergantung implementasi dan biaya akses memori.

Gaussian smoothing juga terkait dengan persamaan difusi $\partial_t u=D\nabla^2u$ pada domain tak berbatas, dengan $\sigma^2=2Dt$ untuk setiap sumbu. **Ini analogi matematis**, bukan pernyataan bahwa kamera mengalami difusi termal.

Untuk konvolusi Gaussian ideal, dua smoothing berturut-turut memiliki varians yang dijumlahkan: $\sigma_{\rm efektif}^2=\sigma_1^2+\sigma_2^2$. Pada implementasi diskret, truncation kernel, padding, dan sampling menyebabkan sedikit deviasi.

In [ ]:
# Verifikasi separability dan penjumlahan varians Gaussian pada data diskret
src=camera
sigx,sigy=1.3,2.1
full=ndimage.gaussian_filter(src,sigma=(sigx,sigy),mode='reflect',truncate=5)
separated=ndimage.gaussian_filter1d(ndimage.gaussian_filter1d(src,sigx,axis=0,mode='reflect',truncate=5),sigy,axis=1,mode='reflect',truncate=5)
print('Kesalahan separabilitas:',np.max(np.abs(full-separated)))
a=ndimage.gaussian_filter(src,sigma=1.0,mode='reflect',truncate=5)
a=ndimage.gaussian_filter(a,sigma=1.5,mode='reflect',truncate=5)
b=ndimage.gaussian_filter(src,sigma=np.sqrt(1.0**2+1.5**2),mode='reflect',truncate=5)
print('Kesalahan komposisi dua Gaussian (diskret):',np.mean(np.abs(a-b)))
fig,axes=plt.subplots(1,3,figsize=(10,3))
for ax,im,cap in zip(axes,[src,full,np.abs(full-separated)],['Awal','Gaussian anisotropik','Galat pemisahan']):
    lihat(ax,im,cap)
plt.tight_layout();plt.show()

## 7. Noise yang berbeda memerlukan asumsi dan strategi berbeda

Tiga model yang sering digunakan:

- **Gaussian baca (read noise)**: $g=\mu+\epsilon$, $\epsilon\sim\mathcal N(0,\sigma_r^2)$; sering menjadi pendekatan terhadap noise elektronik aditif.
- **Shot noise**: jumlah elektron/foton terdeteksi $N\mid\mu\sim\operatorname{Poisson}(\mu)$, sehingga $\operatorname{Var}(N\mid\mu)=\mu$; noise bersifat bergantung sinyal. Untuk intensitas ternormalisasi $N/P$, variansnya $\mu/P$ jika $\mu$ adalah mean yang dinormalisasi dan $P$ skala paparan.
- **Noise impuls**: sebagian piksel tertimpa nilai ekstrem akibat kejadian diskret atau data rusak; Gaussian smoothing bukan strategi khusus untuk model ini.

Sistem nyata juga dapat mengalami *fixed-pattern noise*, korelasi baris/kolom, noise warna, saturasi, dan kuantisasi. Semua model di atas bergantung pada tahap pengolahan: **JPEG terkompresi dan citra yang telah diproses ISP bukan data sensor mentah**. Dalam banyak eksperimen, clipping sebelum menghitung residual dapat mengubah distribusi noise dan membuat model Gaussian tidak lagi tepat.

In [ ]:
# Pisahkan noise Gaussian, shot noise, dan impuls pada citra sama
ref=camera
noise_gauss=np.clip(ref+rng.normal(0,.07,ref.shape),0,1)
photons=180
noise_shot=np.clip(rng.poisson(ref*photons)/photons,0,1)
noise_imp=ref.copy()
rand=rng.random(ref.shape)
noise_imp[rand<.035]=0;noise_imp[(rand>=.035)&(rand<.07)]=1
fig,axes=plt.subplots(1,4,figsize=(13,3))
for ax,im,cap in zip(axes,[ref,noise_gauss,noise_shot,noise_imp],['Referensi','Gaussian (klip)','Poisson (klip)','Impuls (7%)']): lihat(ax,im,cap)
plt.tight_layout();plt.show()
for title,z in [('Gaussian',noise_gauss),('Poisson',noise_shot),('Impuls',noise_imp)]:
    p,s=skor_psnr_ssim(ref,z); print(f'{title:<9} PSNR={p:.2f} dB | SSIM={s:.3f}')

## 8. Filter linier dan nonlinear: memilih berdasarkan struktur noise

**Mean/box filter** menggunakan bobot seragam; murah tetapi mudah meratakan tepi. **Gaussian** mengutamakan tetangga dekat dan mempunyai respons frekuensi halus. **Median** memilih statistik urutan di jendela lokal; sering tangguh terhadap outlier impulsif, tetapi nonlinear dan dapat mengubah struktur tipis. **Bilateral filter** memakai kedekatan spasial *dan* kemiripan intensitas dalam bobotnya:

$$
\widehat f(p)=\frac{\sum_{q\in\Omega_p}w_s(p,q)\,w_r(g(p),g(q))\,g(q)}{\sum_{q\in\Omega_p}w_s(p,q)\,w_r(g(p),g(q))}.
$$

Bobot bergantung pada citra, sehingga bilateral bukan operator konvolusi linear tetap. Pemilihan parameter `sigma_color` terlalu kecil dapat mempertahankan noise, terlalu besar mendekati penghalusan spasial biasa. Pada noise Poisson rendah-foton, transformasi penstabil varians (misalnya Anscombe) dapat membantu pada domain asumsi tertentu, tetapi diperlukan pembahasan bias dan inversnya sebelum digunakan untuk pengukuran.

In [ ]:
# Perbandingan yang adil: tiap noise diperlakukan dan dievaluasi terpisah
from scipy.ndimage import median_filter, uniform_filter

def bandingkan_filter(g,ref):
    out={
        'Mentah':g,
        'Mean 3x3':uniform_filter(g,size=3,mode='reflect'),
        'Gaussian 1':ndimage.gaussian_filter(g,sigma=1.0,mode='reflect'),
        'Median 3x3':median_filter(g,size=3,mode='reflect'),
        'Bilateral':restoration.denoise_bilateral(g,sigma_color=.09,sigma_spatial=2,channel_axis=None),
    }
    tab=[]
    for name,z in out.items():
        psnr,ssim=skor_psnr_ssim(ref,z);tab.append(dict(Metode=name,PSNR_dB=psnr,SSIM=ssim))
    return out,pd.DataFrame(tab)

out_g,tg=bandingkan_filter(noise_gauss,ref)
out_i,ti=bandingkan_filter(noise_imp,ref)
print('Gaussian noise:');display(tg.round(3))
print('Noise impuls:');display(ti.round(3))
fig,axes=plt.subplots(2,5,figsize=(14,5.7))
for row,(out,title) in enumerate([(out_g,'Noise Gaussian'),(out_i,'Noise impuls')]):
    for ax,(name,z) in zip(axes[row],out.items()):lihat(ax,z,title+'\n'+name)
plt.tight_layout();plt.show()

## 9. Metrik dan residu: yang bagus secara visual belum tentu tepat

Dengan citra referensi *diketahui* dan rentang dinamis $L$, *mean squared error* serta PSNR didefinisikan

$$
\operatorname{MSE}=\frac{1}{MN}\sum_{m,n}(f[m,n]-\widehat f[m,n])^2,
\quad
\operatorname{PSNR}=10\log_{10}\frac{L^2}{\operatorname{MSE}}.
$$

PSNR tidak memandang semua kesalahan secara setara dengan persepsi maupun tugas pengukuran. SSIM menilai kesamaan luminansi, kontras, dan struktur lokal menurut parameter jendelanya, tetapi **bukan ukuran akurasi metrologi yang universal**. Jika citra referensi tidak ada, PSNR dan SSIM *full-reference* tidak dapat dihitung secara sah. Dalam inspeksi, kita perlu metrik yang mengikuti tujuan: deteksi cacat, luas area, posisi tepi, atau pergeseran centroid.

Periksa **residu** $\widehat f-f$ untuk membedakan noise yang tersisa dari bias karena penghalusan. Residu dapat berpola di sekitar tepi dan objek kecil, meskipun satu angka PSNR meningkat.

In [ ]:
# PSNR dan SSIM bisa mengabaikan konsekuensi pengukuran geometri
z_g=out_g['Gaussian 1']; z_b=out_g['Bilateral']
roi=np.s_[80:160,65:155]
fig,axs=plt.subplots(2,3,figsize=(11,6))
for ax,im,t in zip(axs[0],[ref[roi],noise_gauss[roi],z_g[roi]],['Referensi','Noisy','Gaussian']):lihat(ax,im,t)
for ax,im,t in zip(axs[1],[z_b[roi],(z_g-ref)[roi],(z_b-ref)[roi]],['Bilateral','Residu Gaussian','Residu bilateral']):
    ax.imshow(im,cmap='gray' if t=='Bilateral' else 'coolwarm',vmin=0 if t=='Bilateral' else -.25,vmax=1 if t=='Bilateral' else .25)
    ax.set_title(t);ax.axis('off')
plt.tight_layout();plt.show()
print('PSNR Gaussian:',round(skor_psnr_ssim(ref,z_g)[0],2),'dB; bilateral:',round(skor_psnr_ssim(ref,z_b)[0],2),'dB')

## 10. PSF, OTF, dan MTF: hubungan filtering dengan resolusi

Untuk sistem linear, invarian pergeseran, dan PSF kontinu $h(x,y)$ terintegrasi satu, OTF adalah transformasi Fourier dari PSF:

$$
H(f_x,f_y)=\iint h(x,y)\exp[-i2\pi(f_xx+f_yy)]\,dx\,dy,
\qquad \operatorname{MTF}(f_x,f_y)=|H(f_x,f_y)|.
$$

OTF bernilai kompleks; fasenya memuat pergeseran/aberrasi asimetris. MTF adalah magnitudonya, bukan seluruh informasi fase. Untuk PSF Gaussian isotropik kontinu simpangan baku $\sigma$ dalam piksel,

$$
\operatorname{MTF}(\nu)=\exp(-2\pi^2\sigma^2\nu^2),
$$

ketika $\nu$ dinyatakan dalam siklus/piksel. Formula ideal ini **bukan** MTF total kamera: apertur piksel, difraksi, *defocus*, interpolasi, dan sampling turut memengaruhi sistem; aliasing muncul ketika pita spasial melampaui Nyquist. Setelah Gaussian smoothing digital, MTF efektif sistem (dalam model LSI) merupakan produk MTF sebelumnya dengan respons kernel filter.

*Sharpening* digital dapat meningkatkan respons amplitudo frekuensi tertentu, tetapi tidak otomatis memulihkan informasi yang sudah hilang di bawah noise atau nilai nol fungsi transfer.

In [ ]:
# PSF dari respons impuls diskret dan perbandingan MTF terhadap aproksimasi kontinu
size=129
imp=np.zeros((size,size),float);imp[size//2,size//2]=1
sigmas=[.8,1.5,2.7]
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
frequency=np.fft.rfftfreq(size,d=1.0)
for sig in sigmas:
    psf=ndimage.gaussian_filter(imp,sigma=sig,mode='constant',truncate=6)
    row=psf[size//2,:]
    mtf=np.abs(np.fft.rfft(np.fft.ifftshift(psf[size//2,:])))
    mtf/=mtf[0]
    analytic=np.exp(-2*np.pi**2*sig**2*frequency**2)
    ax[0].plot(np.arange(-(size//2),size//2+1),row/row.max(),label=f'σ={sig}')
    ax[1].plot(frequency,mtf,lw=2,label=f'Diskret σ={sig}')
    ax[1].plot(frequency,analytic,ls='--',lw=1,alpha=.75)
ax[0].set(xlim=(-12,12),xlabel='Posisi relatif (piksel)',ylabel='PSF ternormalisasi (puncak=1)',title='Penampang PSF')
ax[1].set(xlabel='Frekuensi (siklus/piksel)',ylabel='MTF relatif',title='MTF diskret (solid) vs kontinu (putus)');ax[1].set_xlim(0,.5)
for a in ax:a.legend(fontsize=8);a.grid(alpha=.2)
plt.tight_layout();plt.show()

## 11. Sharpness, unsharp masking, dan amplifikasi noise

Untuk *unsharp masking*, citra keluaran dapat ditulis

$$
u=g+\alpha\bigl(g-(k_\sigma*g)\bigr),\qquad\alpha>0.
$$

Pada domain frekuensi, transfernya $U(\nu)=1+\alpha[1-K_\sigma(\nu)]$. Pada frekuensi tinggi dengan $K_\sigma\approx0$, gain mendekati $1+\alpha$. Jadi filter ini dapat meningkatkan gradien tampak **sekaligus menguatkan noise**. Di dekat tepi tajam dapat muncul *overshoot* dan *ringing-like halos*. Penampakan tepi yang lebih sempit tidak cukup untuk menyimpulkan bahwa PSF optik asli telah dipulihkan.

Dekonvolusi berbeda dari *sharpening*: ia mengestimasi citra laten menggunakan model pembentukan citra dan asumsi/regularisasi. Kita akan membahasnya secara khusus pada materi *inverse problems*.

In [ ]:
# Eksperimen sharpening pada sinyal noisy dan baseline non-noisy
smooth=ndimage.gaussian_filter(noise_gauss,sigma=1.2)
sharpen=noise_gauss+1.6*(noise_gauss-smooth)
sharpen=np.clip(sharpen,0,1)
fig,ax=plt.subplots(1,3,figsize=(10,3))
for a,im,t in zip(ax,[ref,noise_gauss,sharpen],['Referensi','Noisy','Unsharp masked']):lihat(a,im,t)
plt.tight_layout();plt.show()
flat=np.ones((128,128))*0.45
flat_noisy=flat+rng.normal(0,.07,flat.shape)
flat_unsharp=flat_noisy+1.6*(flat_noisy-ndimage.gaussian_filter(flat_noisy,1.2))
for t,img,fpart in [('Noisy',noise_gauss,flat_noisy),('Unsharp',sharpen,flat_unsharp)]:
    p,s=skor_psnr_ssim(ref,img)
    print(t, 'PSNR=',round(p,2),'SSIM=',round(s,3),'varians pada phantom datar=',round(float(fpart[10:-10,10:-10].var()),6))

## 12. Propagasi noise melalui filtering: korelasi tidak boleh diabaikan

Untuk output linier $z=\sum_iw_i x_i$, jika $\mathbf x$ berkovarians $\Sigma$,

$$
\operatorname{Var}(z)=\mathbf w^\mathsf T\Sigma\mathbf w.
$$

Pada noise input independen, identik, dan bervarians $\sigma_n^2$, rumus menyederhana menjadi

$$
\operatorname{Var}(z)=\sigma_n^2\sum_iw_i^2.
$$

Karena bobot rata-rata menjumlah satu, filter smoothing biasanya menekan **varians noise per piksel**, tetapi menggabungkan nilai tetangga dan menciptakan korelasi antarpiksel keluaran. Dua output bersebelahan berbagi banyak sampel input. Pada estimasi mean dari banyak piksel terfilter, menganggap setiap output saling bebas akan **menghitung ketidakpastian terlalu kecil**. Di dekat tepi, smoothing juga menciptakan *bias* terhadap intensitas benar; penurunan varians tidak menghilangkan bias tersebut.

In [ ]:
# Verifikasi varians noise setelah box filter, Gaussian, dan korelasi ketetanggaan
trials=3000; L=128; sig_noise=.09
sim=rng.normal(0,sig_noise,size=(trials,L))
w_box=np.ones(5)/5
filt=ndimage.convolve1d(sim,w_box,axis=1,mode='wrap')
pred_var=sig_noise**2*np.sum(w_box**2)
emp_var=float(filt[:,18].var(ddof=1))
cov_neighbor=float(np.cov(filt[:,18],filt[:,19],ddof=1)[0,1])
print(f'Varians teori per piksel: {pred_var:.6f}; simulasi: {emp_var:.6f}; cov antartetangga: {cov_neighbor:.6f}')
# Untuk boundary periodik, mean output == mean input per realisasi.
means=filt.mean(axis=1)
sd_observed=means.std(ddof=1)
sd_wrong=np.sqrt(pred_var/L) # salah jika menganggap output independen
sd_correct=sig_noise/np.sqrt(L)
print(f'SD mean empiris: {sd_observed:.5f}; prediksi benar: {sd_correct:.5f}; prediksi naif (salah): {sd_wrong:.5f}')
assert abs(emp_var-pred_var)/pred_var < .10
assert abs(sd_observed-sd_correct)/sd_correct < .10

## 13. Bias–variance trade-off dan pemilihan parameter filter

Pada piksel atau parameter pengukuran $\theta$, estimasi $\widehat\theta$ memiliki galat kuadrat rata-rata

$$
\operatorname{MSE}(\widehat\theta)=\operatorname{Bias}(\widehat\theta)^2+\operatorname{Var}(\widehat\theta)
$$

untuk distribusi sampling yang jelas. Smoothing lebih agresif dapat mengurangi varians tetapi memperbesar bias ketika tepi/objek kecil teredam. Lebar kernel dan `sigma` bukan parameter kosmetik; keduanya mengatur *spatial bandwidth*, resolusi, dan respons terhadap noise.

Untuk memilih parameter secara sahih: tentukan metrik berdasarkan tujuan, pilih parameter menggunakan **data validasi**, bekukan keputusan, lalu laporkan performa pada **spesimen uji terpisah**. Jika parameter dipilih dari data uji, hasil uji tidak lagi independen. Dalam penelitian sesungguhnya, pisahkan juga batch, perangkat, atau lokasi ketika hal itu menjadi sumber perbedaan domain.

In [ ]:
# Bias global di tepi dan perbaikan MSE keseluruhan dapat bergerak berlawanan
one=np.zeros((160,160),float)+.65
one[:,79:]=.2
noisy_one=np.clip(one+rng.normal(0,.08,one.shape),0,1)
records=[]
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
for sg in [0,0.6,1.2,2.2,4.0]:
    z=noisy_one if sg==0 else ndimage.gaussian_filter(noisy_one,sg)
    mse=float(np.mean((z-one)**2))
    edge_bias=float(np.mean(np.abs(z[:,77:82]-one[:,77:82])))
    records.append((sg,mse,edge_bias))
    ax[0].plot(z[80,65:95],label=f'σ={sg}')
rows=np.array(records)
ax[0].plot(one[80,65:95],'k--',lw=2,label='Referensi');ax[0].legend(fontsize=8);ax[0].set(xlabel='Posisi lokal',ylabel='Intensitas',title='Penampang melintasi tepi');ax[0].grid(alpha=.2)
ax[1].plot(rows[:,0],rows[:,1],'-o',label='MSE seluruh citra');ax[1].plot(rows[:,0],rows[:,2],'-s',label='Bias absolut sekitar tepi')
ax[1].legend(fontsize=8);ax[1].set(xlabel='Sigma (piksel)',ylabel='Besaran galat',title='Fungsi objektif berubah menurut tugas');ax[1].grid(alpha=.2)
plt.tight_layout();plt.show()

## 14. Studi kasus integratif — Inspeksi cacat permukaan dengan model sensor

**Masalah engineering.** Sebuah sistem inspeksi optik memeriksa material yang mempunyai daerah cacat lebih gelap daripada substrat. Kamera mengalami *blur* PSF, variasi gain/iluminasi, shot noise, read noise, dan sedikit noise impuls. Insinyur ingin menghitung **luas cacat** dari hasil segmentasi, bukan sekadar menghasilkan citra yang tampak bersih.

Kita membangun **dataset sintetis parametrik**: setiap spesimen memiliki satu cacat elips dengan posisi dan ukuran berbeda; `ground_truth` tersedia karena bentuk objek ditentukan secara analitis. Satu spesimen merupakan satu unit eksperimen. Dataset dibagi menjadi **9 spesimen validasi** untuk memilih parameter/ambang dan **13 spesimen uji terpisah** untuk laporan akhir. Tidak ada *training* ML karena filter diuji sebagai algoritma deterministik; pemilihan parameternya tetap bentuk optimisasi dan harus dibatasi pada validasi.

**Metrik:** F1 piksel untuk segmentasi, kesalahan luas dalam mm² berdasarkan kalibrasi skala **hipotetis**, PSNR terhadap *latent clean image*, serta variasi hasil antarspesimen. Pelaporan harus membedakan noise observasi, blur sebelum filtering, dan bias segmentasi; PSNR ke citra laten mencampur dampak blur serta denoising dan bukan metrik restorasi tanpa caveat.

**Batas klaim:** tidak ada kalibrasi kamera nyata, tidak ada estimasi ukuran fisik dari metrologi lapangan, dan tidak ada klaim transfer ke berbagai batch industri.

In [ ]:
# Generator data sintetis reproducible, tanpa file eksternal
H=W=144
pitch_mm=.12  # mm/piksel: ASUMSI SIMULASI, bukan hasil kalibrasi instrumen
Y,X=np.mgrid[:H,:W]

def simulasi_spesimen(seed):
    r=np.random.default_rng(seed)
    cx=r.uniform(47,97);cy=r.uniform(48,96)
    rx=r.uniform(9,17);ry=r.uniform(7,14)
    theta=r.uniform(-.6,.6)
    u=(X-cx)*np.cos(theta)+(Y-cy)*np.sin(theta)
    v=-(X-cx)*np.sin(theta)+(Y-cy)*np.cos(theta)
    defect=((u/rx)**2+(v/ry)**2)<=1
    # Latar tidak seragam di bawah asumsi sinyal sensor linear
    base=.63+.025*(X/W-.5)+.016*np.sin(2*np.pi*Y/H)
    clean=base-r.uniform(.19,.28)*defect.astype(float)
    blurred=ndimage.gaussian_filter(clean,sigma=r.uniform(.8,1.4),mode='reflect')
    photons=r.uniform(230,460)
    shot=r.poisson(np.maximum(blurred,0)*photons)/photons
    observed=shot+r.normal(0,r.uniform(.014,.024),size=shot.shape)
    impulse=r.random(shot.shape)
    observed=np.where(impulse<.003,0,np.where(impulse>.997,1,observed))
    observed=np.clip(observed,0,1)
    return {'clean':clean,'blurred':blurred,'observed':observed,'mask':defect,
            'true_area_mm2':float(defect.sum()*pitch_mm**2),'seed':seed}

validation=[simulasi_spesimen(1000+i) for i in range(9)]
test=[simulasi_spesimen(2000+i) for i in range(13)]
fig,ax=plt.subplots(1,4,figsize=(12,3))
s=validation[0]
for a,z,t in zip(ax,[s['clean'],s['blurred'],s['observed'],s['mask'].astype(float)],['Citra laten','Setelah PSF','Sensor sintetis','Mask ground truth']):lihat(a,z,t)
plt.tight_layout();plt.show()
print('Validasi:',len(validation),'spesimen | Uji:',len(test),'spesimen | Skala asumsi:',pitch_mm,'mm/piksel')

### 14A. Kandidat filter dan definisi tugas segmentasi

Agar perbandingan tidak dipengaruhi kemampuan mengetahui label uji, kita hanya menggunakan mask pada spesimen validasi untuk memilih konfigurasi. Objek cacat diasumsikan lebih gelap; segmentasi dilakukan dengan threshold intensitas $I<t$. Tidak ada *postprocessing* morfologi di sini agar sifat masing-masing filter tidak tertutup prosedur tambahan.

Kandidat: tanpa filter, Gaussian dengan beberapa sigma, median dengan beberapa ukuran jendela, dan bilateral dengan konfigurasi tetap. Nilai ambang dipilih bersama parameter filter pada **validation set**. Skor dihitung **per spesimen**, lalu dirata-ratakan untuk mencegah spesimen lebih besar mendominasi tujuan optimisasi tanpa justifikasi.

In [ ]:
# Optimisasi hyperparameter + threshold hanya di validation set
candidates={
    'Mentah': [('tanpa',None)],
    'Gaussian': [('sigma',.6),('sigma',1.1),('sigma',1.8)],
    'Median': [('ukuran',3),('ukuran',5)],
    'Bilateral': [('sigma_color',.06),('sigma_color',.11)],
}
thresholds=np.arange(.39,.581,.01)

def apply_filter(image,method,parameter):
    if method=='Mentah':return image
    if method=='Gaussian':return ndimage.gaussian_filter(image,parameter[1],mode='reflect')
    if method=='Median':return ndimage.median_filter(image,size=parameter[1],mode='reflect')
    if method=='Bilateral':return restoration.denoise_bilateral(image,sigma_color=parameter[1],sigma_spatial=2,channel_axis=None)
    raise ValueError(method)

def f1_mask(pred,true):
    intersection=np.count_nonzero(pred & true)
    denom=np.count_nonzero(pred)+np.count_nonzero(true)
    return 2*intersection/denom if denom else 1.0

def val_score(method,param,threshold):
    return np.mean([f1_mask(apply_filter(s['observed'],method,param)<threshold,s['mask']) for s in validation])

chosen={}
for method,parameters in candidates.items():
    # Cache citra validasi agar grid threshold tidak memproses filter berulang
    best=(-np.inf,None,None)
    for param in parameters:
        ims=[apply_filter(s['observed'],method,param) for s in validation]
        for threshold in thresholds:
            score=float(np.mean([f1_mask(im<threshold,s['mask']) for im,s in zip(ims,validation)]))
            if score>best[0]+1e-12:best=(score,param,float(threshold))
    chosen[method]={'val_f1':best[0],'param':best[1],'threshold':best[2]}
print('KONFIGURASI TERPILIH PADA VALIDASI (tanpa melihat mask test)')
display(pd.DataFrame([dict(Metode=k,Parameter=str(v['param']),Ambang=v['threshold'],F1_validasi=v['val_f1']) for k,v in chosen.items()]).round(4))

### 14B. Evaluasi *held-out test*: deteksi dan pengukuran area

Setiap metode kini menggunakan konfigurasi dari validasi tanpa perubahan. Untuk spesimen $j$, luas estimasi adalah $\widehat A_j=N_{\rm pred,j}p^2$, dengan $p$ skala mm/piksel yang *diasumsikan* dalam simulasi. Kesalahan area ditulis $\widehat A_j-A_j$ dan $|\widehat A_j-A_j|$. Kesalahan segmentasi pada batas objek kecil dapat memiliki dampak besar terhadap luas walaupun PSNR citra meningkat.

Kesalahan relatif dapat tidak stabil untuk objek yang sangat kecil; dataset simulasi ini memiliki objek yang cukup besar. Dalam aplikasi nyata, ketidakpastian $p$ akibat kalibrasi menambah komponen ketidakpastian terpisah: jika $A=Np^2$, pendekatan orde pertama terhadap skala adalah $\delta A/A\approx2\delta p/p$ ketika $N$ tetap.

In [ ]:
rows=[]
for method,setting in chosen.items():
    for j,s in enumerate(test):
        filtered=apply_filter(s['observed'],method,setting['param'])
        pred=filtered<setting['threshold']
        pred_area=float(pred.sum()*pitch_mm**2)
        psnr,ssim=skor_psnr_ssim(s['clean'],filtered)
        rows.append({'metode':method,'spesimen':j,'f1':f1_mask(pred,s['mask']),
                     'pred_area_mm2':pred_area,'real_area_mm2':s['true_area_mm2'],
                     'abs_area_error_mm2':abs(pred_area-s['true_area_mm2']),
                     'area_bias_mm2':pred_area-s['true_area_mm2'],'psnr_db':psnr,'ssim':ssim})
results=pd.DataFrame(rows)
summary=results.groupby('metode').agg(F1_mean=('f1','mean'),F1_sd=('f1','std'),
    MAE_luas_mm2=('abs_area_error_mm2','mean'),Bias_luas_mm2=('area_bias_mm2','mean'),
    PSNR_dB=('psnr_db','mean'),SSIM=('ssim','mean')).sort_values('F1_mean',ascending=False)
print('HASIL SPESIMEN UJI (konfigurasi dibekukan setelah validasi)')
display(summary.round(3))
fig,ax=plt.subplots(1,2,figsize=(10,3.7))
summary.F1_mean.plot.bar(ax=ax[0],ylim=(0,1));ax[0].set(ylabel='F1 rerata',title='Segmentasi pada 13 spesimen');ax[0].tick_params(axis='x',rotation=20)
summary.MAE_luas_mm2.plot.bar(ax=ax[1]);ax[1].set(ylabel='MAE luas (mm², skala asumsi)',title='Kesalahan metrologi geometri');ax[1].tick_params(axis='x',rotation=20)
plt.tight_layout();plt.show()

### 14C. Interval kepercayaan bersyarat dan unit replikasi

Untuk meringkas variasi F1 antarspesimen uji, lakukan *nonparametric bootstrap* atas **spesimen**, bukan atas piksel. Resampling piksel akan menganggap piksel independen meskipun noise terfilter berkorelasi dan nilai label membentuk satu objek. Interval 95% yang dihasilkan adalah interval *bootstrap percentile* untuk rerata pada **populasi simulasi yang didefinisikan generator**, bukan jaminan kinerja di industri.

Karena hanya ada 13 spesimen uji, interval bisa sensitif terhadap distribusi sampling dan tidak memperhitungkan kesalahan model simulator maupun variasi sensor nyata. Jangan memilih metode pemenang lalu menyatakan perbedaan signifikan hanya karena interval ringkasan tampak berbeda; bandingkan pasangan per spesimen, laporkan ukuran efek, dan rancangkan eksperimen konfirmasi yang independen.

In [ ]:
# CI bootstrap untuk mean F1 di tingkat spesimen; seed terkontrol
boot_rng=np.random.default_rng(SEED+17)
ci_records=[]
for method,group in results.groupby('metode'):
    values=group.sort_values('spesimen')['f1'].to_numpy()
    resampled=boot_rng.choice(values,size=(4000,len(values)),replace=True).mean(axis=1)
    lo,hi=np.percentile(resampled,[2.5,97.5])
    ci_records.append({'Metode':method,'Rerata F1':values.mean(),'CI 95% bawah':lo,'CI 95% atas':hi})
ci=pd.DataFrame(ci_records).sort_values('Rerata F1',ascending=False)
display(ci.round(3))

### 14D. Diagnosis kegagalan: inspeksi visual tidak menggantikan analisis residual

Periksa satu spesimen uji yang **tidak dipakai** untuk menyetel threshold. Visualisasikan hasil segmentasi dan residu klasifikasi: *false positive* (latar ditandai cacat) dan *false negative* (cacat terlewat). Objek yang tertinggal setelah smoothing atau meluas akibat blur dapat menghasilkan galat area sistematis.

Interpretasi harus membedakan: (1) blur sensor sebelum pemrosesan, (2) filter pascaakuisisi, (3) threshold yang dilatih pada validasi, dan (4) perubahan bentuk objek setelah threshold. **Mask sintetis merupakan kebenaran geometris simulator**, bukan anotasi ahli yang memiliki ketidakpastian pengamat.

In [ ]:
case=test[4]
fig,axs=plt.subplots(2,5,figsize=(14,5.7))
for i,(method,setting) in enumerate(chosen.items()):
    z=apply_filter(case['observed'],method,setting['param'])
    pred=z<setting['threshold']
    err=np.zeros(case['mask'].shape,int)
    err[pred & ~case['mask']]=1 # false positive
    err[~pred & case['mask']]=-1 # false negative
    lihat(axs[0,i],z,method)
    axs[1,i].imshow(err,cmap='bwr',vmin=-1,vmax=1);axs[1,i].set_title('Biru: FN | Merah: FP');axs[1,i].axis('off')
lihat(axs[0,-1],case['mask'].astype(float),'Ground truth')
lihat(axs[1,-1],case['observed'],'Sensor mentah')
fig.suptitle('Satu spesimen uji: noise terfilter dan kesalahan segmentasi',y=1.02)
plt.tight_layout();plt.show()

## 15. Propagasi ketidakpastian menuju pengukuran ilmiah

Filter adalah bagian dari rantai instrumentasi, bukan sekadar tahap mempercantik gambar. Ketidakpastian hasil pengukuran dapat berasal dari: variasi foton, sensor, pemilihan threshold, model PSF, bias segmentasi, dan kalibrasi panjang. Pada pengukuran $A=Np^2$, ketidakpastian jumlah piksel $N$ (yang bergantung pada mask dan threshold) tidak identik dengan fluktuasi noise per piksel citra mentah.

**Metode delta** untuk fungsi $A(N,p)=Np^2$, dengan kovarians variabel $(N,p)$, menghasilkan

$$
\operatorname{Var}(A)\approx
p^4\operatorname{Var}(N)
+4N^2p^2\operatorname{Var}(p)
+4Np^3\operatorname{Cov}(N,p).
$$

Jika $N$ adalah hasil diskret dari threshold, kelancaran fungsi yang diasumsikan metode delta mungkin tidak cocok untuk propagasi langsung dari intensitas piksel ke jumlah objek. Monte Carlo pada keseluruhan pipeline lebih defensibel ketika ketidaklinieran dan perubahan topologi mask dominan. Dalam simulator berikut, kita hanya mengilustrasikan sensitivitas luas terhadap ketidakpastian skala—bukan mengestimasi seluruh *uncertainty budget* instrumentasi.

In [ ]:
# Sensitivitas terhadap kalibrasi piksel (contoh hipotetis)
N_pix=int(case['mask'].sum())
p0=pitch_mm
sigma_p=.0015 # mm/piksel, asumsi ilustratif
sigma_A_linear=2*N_pix*p0*sigma_p
samples_p=np.random.default_rng(SEED+21).normal(p0,sigma_p,size=30000)
area_sim=N_pix*samples_p**2
print('Jumlah piksel mask referensi:',N_pix)
print(f'Luas dengan p={p0} mm/piksel: {N_pix*p0**2:.3f} mm²')
print(f'SD luas dari propagasi linear: {sigma_A_linear:.3f} mm²')
print(f'SD luas dari Monte Carlo kalibrasi saja: {area_sim.std(ddof=1):.3f} mm²')
assert abs(area_sim.std(ddof=1)-sigma_A_linear)/sigma_A_linear < .05

## 16. Verifikasi numerik dan reproducibility

Sebelum menganggap eksperimen valid, lakukan uji invarians dan konsistensi: kernel perataan menjumlah satu, konvolusi manual cocok dengan SciPy, Gaussian separable cocok dengan implementasi 2D, prediksi varians noise sesuai simulasi dalam toleransi statistik, dan daftar hasil uji mempunyai satu baris untuk setiap metode–spesimen. Pengujian kode tidak menggantikan uji sensor nyata, tetapi dapat mendeteksi kesalahan implementasi dan asumsi numerik.

In [ ]:
# Uji numerik cepat; bisa dijalankan di Colab tanpa jaringan setelah dependency tersedia
assert np.isclose(ker.sum(),1)
assert np.allclose(naive,conv)
assert err_linear < 1e-12
assert err_shift < 1e-12
assert err_median > 1e-4
assert np.max(np.abs(full-separated)) < 1e-12
assert len(validation)==9 and len(test)==13
assert len(results)==len(test)*len(chosen)
assert results['f1'].between(0,1).all()
assert results['abs_area_error_mm2'].ge(0).all()
assert all(set(v)=={'val_f1','param','threshold'} for v in chosen.values())
assert all(np.isfinite(results[c]).all() for c in ['f1','psnr_db','ssim'])
assert np.all(results['spesimen'].isin(np.arange(len(test))))
print('Semua pemeriksaan numerik dan integritas eksperimen lulus.')

## 17. Audit asumsi dan interpretasi ilmiah

1. **Domain data:** apakah filter diterapkan pada berkas JPEG, nilai kode RAW linear, atau intensitas/radiansi yang telah dikalibrasi?
2. **Model sistem:** apakah PSF benar-benar tidak berubah terhadap posisi dan panjang gelombang? Apakah gerak objek mengubahnya?
3. **Batas:** apakah ekstensi `reflect` mewakili nilai di luar bidang pengamatan? Jika tidak, berapa besar ROI terdampak?
4. **Noise:** apakah simpangan baku sama untuk semua intensitas? Apakah ada korelasi baris atau shot noise dominan?
5. **Tujuan:** apakah kita mengoptimalkan persepsi, PSNR, deteksi, atau galat luas? Mengapa bisa menghasilkan peringkat yang berbeda?
6. **Metrologi:** apakah mm/piksel dikalibrasi? Apakah galat PSF, threshold, dan scale masuk *uncertainty budget*?
7. **Validasi:** apakah parameter dipilih hanya pada spesimen validasi? Apakah spesimen uji benar-benar independen dalam generator data?
8. **Generalisasi:** kondisi simulator mana yang belum mencakup kamera baru, eksposur berubah, bahan lain, dan batch baru?
9. **Klaim:** apakah kesimpulan dibatasi pada eksperimen sintetis? Apakah ada ketidakpastian sampling pada metrik hasil test?

**Prinsip:** citra yang lebih halus tidak berarti informasi pengukuran bertambah; bahkan kadang menghapus tepi yang dibutuhkan untuk estimasi geometris.

## 18. Latihan bertingkat, proyek mini, dan HOTS

**Latihan 1 — Konvolusi (dasar).** Hitung konvolusi sebuah matriks $5\times5$ dengan kernel asimetris $3\times3$ pada posisi pusat secara manual. Ulangi sebagai korelasi. Jelaskan tanda dan pembalikan kernel.

**Latihan 2 — Sistem linear (analitis).** Buktikan bahwa konvolusi dengan kernel tetap merupakan operator linear. Berikan contoh numerik sederhana yang menunjukkan bahwa median filter tidak linear. Jelaskan kaitan invarian pergeseran dengan boundary condition.

**Latihan 3 — Gaussian dan MTF (analitis).** Turunkan transformasi Fourier PSF Gaussian satu dimensi, kemudian generalisasikan ke isotropik dua dimensi. Untuk $\sigma=1.5$ piksel, tentukan frekuensi saat MTF turun menjadi 0.5 dan interpretasikan dalam siklus/piksel. Bandingkan dengan Nyquist.

**Latihan 4 — Model noise (komputasional).** Pada intensitas seragam $\mu=0.2,0.5,0.8$, simulasi shot noise dan read noise. Plot varians terhadap mean sebelum dan setelah Gaussian filter. Jelaskan kapan pendekatan varians konstan gagal.

**Latihan 5 — Propagasi noise (HOTS).** Sebuah kernel 1D mempunyai bobot $(1/4,1/2,1/4)$, noise input iid dengan $\sigma=0.08$. Hitung varians satu piksel output, kovarians dua piksel output bertetangga, dan jelaskan mengapa varians mean banyak output tidak bisa dihitung dengan mengabaikan kovarians.

**Latihan 6 — Metrologi (HOTS).** Ubah ukuran cacat menjadi 3–6 piksel dan naikkan blur hingga $\sigma=2.5$. Evaluasi apakah metode dengan PSNR tertinggi juga memberikan galat luas terkecil. Bandingkan *false negative* dan bias area. Jelaskan mengapa *sampling* dan PSF membatasi ukuran cacat yang dapat diukur.

**Latihan 7 — Desain eksperimen (HOTS).** Bagi data menurut *batch simulator*: validasi pada $P=450$ foton/skala dan test pada $P=120$, dengan perubahan PSF serta noise. Tanpa menyetel ulang parameter pada test, periksa perubahan ranking filter. Susun protokol uji silang antarperangkat untuk aplikasi nyata.

**Latihan 8 — Inverse problem (HOTS).** Jelaskan mengapa filter sharpening yang menaikkan gradien bukan solusi yang sama dengan dekonvolusi teregularisasi. Berikan contoh saat $|H(\nu)|$ mendekati nol dan konsekuensinya terhadap amplifikasi noise.

**Output proyek mini:** (i) persoalan, (ii) hipotesis, (iii) model sensor dan asumsi, (iv) skema eksperimen, (v) dataset dan seed, (vi) validasi parameter, (vii) hasil dengan metrik tugas, (viii) uncertainty/bias analysis, (ix) failure cases, dan (x) batas klaim.

## 19. Rangkuman dan koneksi berikutnya

1. Histogram dan transformasi intensitas menjelaskan distribusi nilai; filtering spasial memerlukan struktur dan hubungan ketetanggaan piksel.
2. Konvolusi berlaku bagi sistem linear invarian pergeseran; korelasi, batas citra, dan operator nonlinear mempunyai perilaku berbeda.
3. Kernel Gaussian, median, dan bilateral mengubah sinyal dengan mekanisme berbeda; keunggulannya bergantung noise dan tujuan.
4. PSF adalah representasi spasial respons sistem; OTF/MTF menjelaskan transfer komponen frekuensi dan keterbatasan resolusi.
5. Filtering dapat menekan varians noise lokal sekaligus menciptakan korelasi antaroutput dan bias bentuk objek.
6. Sharpening bisa memperkuat tepi tampak namun juga noise; dekonvolusi membutuhkan model inverse problem dan regularisasi.
7. Evaluasi yang sahih memisahkan validasi parameter dari test dan melaporkan metrik yang sesuai dengan pengukuran, bukan satu angka PSNR saja.
8. Simulasi instrumen dan kalibrasi skala yang diasumsikan tidak setara dengan validasi metrologi sensor nyata.

**Koneksi berikutnya:** deteksi tepi memanfaatkan respons spasial dan gradien untuk menemukan batas objek; segmentasi mengubah respons tersebut menjadi keputusan geometri; domain frekuensi memperdalam interpretasi PSF/OTF dan batas bandwidth; *inverse problems* membahas rekonstruksi di bawah noise dan informasi hilang; pengukuran ilmiah meneruskan analisis bias, kovarians, kalibrasi, serta ketidakpastian luas/posisi objek.

**Pertanyaan penutup:** apabila filter berhasil mengurangi standar deviasi noise sebesar 50%, mengapa insinyur belum boleh menyatakan ketidakpastian pengukuran luas objek berkurang sebesar 50%?

## 20. Bacaan lanjutan, dokumentasi, dan provenance

- Gonzalez, R. C. & Woods, R. E., *Digital Image Processing*, edisi ke-4, Pearson (2018), bagian spatial filtering, image restoration, dan frequency-domain processing.
- Szeliski, R., *Computer Vision: Algorithms and Applications*, edisi ke-2, Springer (2022), bab image processing dan image formation. https://szeliski.org/Book/
- Burger, W. & Burge, M. J., *Digital Image Processing: An Algorithmic Introduction Using Java*, edisi ke-3, Springer (2022), bagian filters and spatial operations.
- Tomasi, C. & Manduchi, R. (1998), *Bilateral Filtering for Gray and Color Images*, ICCV. https://doi.org/10.1109/ICCV.1998.710815
- Wang, Z., Bovik, A. C., Sheikh, H. R. & Simoncelli, E. P. (2004), *Image Quality Assessment: From Error Visibility to Structural Similarity*, IEEE Transactions on Image Processing. https://doi.org/10.1109/TIP.2003.819861
- SciPy `ndimage` dan `signal`: https://docs.scipy.org/doc/scipy/reference/ndimage.html ; https://docs.scipy.org/doc/scipy/reference/signal.html
- scikit-image filters, restoration, dan metrics: https://scikit-image.org/docs/stable/api/skimage.filters.html ; https://scikit-image.org/docs/stable/api/skimage.restoration.html ; https://scikit-image.org/docs/stable/api/skimage.metrics.html
- JCGM 100:2008 (*Guide to the Expression of Uncertainty in Measurement*), untuk konsep propagasi kovarians dan *uncertainty budget*: https://www.bipm.org/en/committees/jc/jcgm/publications

**Batas sumber:** seluruh angka pada grafik, tabel, dan pengukuran studi kasus berasal dari pelaksanaan kode simulator dalam notebook. Tidak ada nilai yang dipresentasikan sebagai hasil pengukuran fisik, hasil dataset industri, atau kutipan performa dari referensi di atas.